# Take Home Exercise 2 
Sumtung Lo <br>sl5919<br>Apri 06 2026

In [0]:
from pyspark.sql import functions as F

In [0]:
df_pit_stops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)

In [0]:
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)

In [0]:
df_pit_stops = df_pit_stops.withColumn("duration", F.col("duration").cast("double"))

In [0]:
from pyspark.sql import functions as F
df_pit_stops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
df_races     = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv",     header=True)

# Parse "mm:ss.mmm" or "ss.mmm" → total seconds (no direct cast to double)
df_pit_stops = df_pit_stops.withColumn(
    "duration_sec",
    F.when(
        F.col("duration").contains(":"),
        F.split(F.col("duration"), ":").getItem(0).cast("double") * 60 +
        F.split(F.col("duration"), ":").getItem(1).cast("double")
    ).otherwise(
        F.col("duration").cast("double")
    )
)


In [0]:
# Parse "mm:ss.mmm" or "ss.mmm" → total seconds (no direct cast to double)
df_pit_stops = df_pit_stops.withColumn(
    "duration_sec",
    F.when(
        F.col("duration").contains(":"),
        F.split(F.col("duration"), ":").getItem(0).cast("double") * 60 +
        F.split(F.col("duration"), ":").getItem(1).cast("double")
    ).otherwise(
        F.col("duration").cast("double")
    )
)


In [0]:
df = df_pit_stops.join(
    df_races.select("raceId", "year", "name"),
    on="raceId",
    how="left"
)

In [0]:
df_summary = (
    df.groupBy("raceId", "year", "name")
      .agg(
          F.round(F.avg("duration_sec"), 3).alias("avg_pit_stop_sec"),
          F.round(F.min("duration_sec"), 3).alias("fastest_pit_stop_sec"),
          F.round(F.max("duration_sec"), 3).alias("slowest_pit_stop_sec"),
          F.count("*").alias("total_pit_stops"),
      )
      .orderBy("year", "name")
)


In [0]:
display(df_summary)